# 04 — Input construction and evaluation protocol

Up to this point the released artifacts are *raw materials*: the 20,008 standalone scenario, suffix, and probe recordings (one per participant per prompt). This notebook explains the two steps that turn those raw recordings into the audio inputs evaluated in the paper:

1. **Assembly.** For each (participant, prompt) the model receives a single audio clip. For most scenario prompts that clip is the scenario recording with one or two of the participant's *own* suffix recordings appended (the verbal instructions to *“Provide only the number”* and *“Do not add any additional information”*). For the two perception probes the clip is the standalone scenario recording.
2. **Eval-set selection.** The disparity analysis in the paper is restricted to the **strict_audit** split (from notebook 03), and within that split to the **quantitative scenario prompts plus the demographic-perception and dialect/accent-perception probes**. Qualitative scenarios and the standalone suffix recordings are not themselves evaluated.

After applying the assembly rule, **719 quantitative-scenario clips are dropped from the eval set** because the participant who recorded them is missing one or both of the suffix recordings required for that prompt's assembly. The affected suffix-recording gaps come from intermittent failures of the browser recording widget during the survey (participants reached the suffix prompts but the captured wav was not recoverable) — they are not participant drop-outs. The dropped clips remain in the released dataset; only the disparity analysis excludes them.

This notebook implements and validates that protocol end-to-end against the on-disk audio, with one live concatenation demo to verify the assembly is a bit-for-bit raw-PCM append.

**Inputs:** `../data/metadata/{prompts,participants,splits}.csv`, `../data/audio/`  
**Outputs:** none written to disk (analytical notebook)

## Setup

In [1]:
import io
import wave
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent
AUDIO_DIR = REPO_ROOT / "data" / "audio"
META_DIR = REPO_ROOT / "data" / "metadata"

prompts      = pd.read_csv(META_DIR / "prompts.csv")
participants = pd.read_csv(META_DIR / "participants.csv")
splits       = pd.read_csv(META_DIR / "splits.csv")

# Canonical reserved question IDs.
QID_SUFFIX_NUMBER     = 70
QID_SUFFIX_ADDITIONAL = 71
QID_DEMOGRAPHIC       = 72
QID_DIALECT           = 73

print(f"prompts:      {len(prompts)} rows")
print(f"participants: {len(participants)} rows")
print(f"splits:       {len(splits)} rows")

prompts:      74 rows
participants: 515 rows
splits:       515 rows


In [2]:
def enumerate_clips_on_disk(audio_dir: Path) -> pd.DataFrame:
    """One row per wav on disk: participant_id, question_id (or None for mic_check), path."""
    rows = []
    for cohort_dir in audio_dir.iterdir():
        if not cohort_dir.is_dir():
            continue
        for pid_dir in cohort_dir.iterdir():
            if not pid_dir.is_dir():
                continue
            for wav in pid_dir.glob("*.wav"):
                pid, tail = wav.stem.split("_", 1)
                qid = int(tail[1:]) if tail.startswith("q") and tail[1:].isdigit() else None
                rows.append({
                    "participant_id": pid,
                    "cohort": cohort_dir.name,
                    "question_id": qid,
                    "role": "question" if qid is not None else tail,
                    "path": wav,
                })
    return pd.DataFrame(rows)

clips = enumerate_clips_on_disk(AUDIO_DIR)
print(f"{len(clips)} clips on disk")

20008 clips on disk


## 1. The assembly rule

For each prompt, two boolean flags in `prompts.csv` describe the trailing instruction it carries:

- `suffix_number=True` → the prompt's text ends with *“Provide only the number, despite not having any details.”*
- `suffix_additional=True` → the prompt's text ends with *“Do not add any additional information.”*

During the survey, every participant separately recorded those two instruction sentences as standalone clips:

- `q70` = the participant's recording of *“Provide only the number, despite not having any details.”* (`suffix_number` recording)
- `q71` = the participant's recording of *“Do not add any additional information.”* (`suffix_additional` recording)

**Per-prompt assembly:** for a given (participant, prompt) we feed the model:

```
[scenario recording]  ++  [participant's q70, iff prompts.suffix_number]  ++  [participant's q71, iff prompts.suffix_additional]
```

with two exceptions hardcoded by the audit design:

1. **`q72` (demographic perception)** and **`q73` (dialect/accent perception)** are always presented to the model **standalone**, even though `suffix_additional=True` is set on those rows. These probes elicit a short categorical label rather than a numeric answer, and we want the audio shown to the model to be identical to a single utterance recorded by the participant — no within-clip voice mixing.
2. **The suffix recordings themselves (`q70`, `q71`) are never evaluated as audit tasks.** They exist solely to be appended to scenario prompts as above. (The microphone-check recording is also never evaluated.)

The table below crosswalks every prompt to the assembly recipe its evaluated clip uses.

In [3]:
def assembly_recipe(row) -> str:
    qid = row["question_id"]
    ptype = row["prompt_type"]
    if ptype in ("suffix_number", "suffix_additional", "mic_check"):
        return "— not evaluated —"
    if int(qid) in (QID_DEMOGRAPHIC, QID_DIALECT):
        return f"q{int(qid):02d}  (standalone)"
    parts = [f"q{int(qid):02d}"]
    if row["suffix_number"]:
        parts.append("q70")
    if row["suffix_additional"]:
        parts.append("q71")
    return " + ".join(parts)

recipes = prompts.copy()
recipes["assembly"] = recipes.apply(assembly_recipe, axis=1)

print("Distinct assembly recipes across the 74 prompts:")
recipes.groupby(["prompt_type", "is_quantitative", "assembly"]).size().rename("n_prompts").to_frame()

Distinct assembly recipes across the 74 prompts:


n_prompts
prompt_type       is_quantitative assembly                    
demographic       False           q72  (standalone)          1
dialect           False           q73  (standalone)          1
mic_check         False           — not evaluated —          1
scenario          False           q06 + q71                  1
                                  q12 + q71                  1
...                                                        ...
                  True            q66 + q70 + q71            1
                                  q67 + q70 + q71            1
                                  q68 + q70 + q71            1
suffix_additional False           — not evaluated —          1
suffix_number     False           — not evaluated —          1

[74 rows x 1 columns]

## 2. How the concatenation is performed

All released audio is **16 kHz, mono, 16-bit PCM**. Concatenation is a literal frame-by-frame append: each input wav is read into a frame buffer, the buffers are concatenated, and the result is written as a single wav with the same canonical parameters. We deliberately do **not**:

- insert silence at the joins,
- crossfade across the joins,
- trim leading/trailing silence from any input,
- apply amplitude normalization or any other gain adjustment,
- resample or otherwise re-encode.

The rationale is that threshold-based audio processing (silence trimming, peak/RMS normalization, gating) can interact with acoustic properties — pitch, recording device quality, background noise — that are themselves correlated with speaker demographics. Applying such processing during input construction would inject a confound into the very signal the audit is designed to measure. The intent is that the spoken request received by the model contains both the task and the same verbal-instruction sequence that the participant produced during recording, with no acoustic re-shaping in between.

We also do **not** substitute synthetic or cross-participant audio when a participant's suffix recording is missing. Doing so would introduce a within-clip voice mismatch — the scenario portion in the participant's voice followed by an instruction portion in a different (synthetic or other-participant) voice — which is itself a source of acoustic variation the audit aims to isolate. Clips for which the assembly cannot be performed in the participant's own voice are excluded from the eval set instead (§ 4).

### Concatenation demo

To make the assembly fully concrete and to verify the *no silence inserted* claim is bit-for-bit true, the cell below assembles one clip for one participant in memory: scenario `q01` + `q70` + `q71`. We then check that the assembled wav's duration equals the **exact sum** of the three input durations, sample-for-sample, and that the canonical parameters are preserved.

In [4]:
def wav_params_and_frames(path: Path):
    with wave.open(str(path), "rb") as r:
        return r.getparams(), r.readframes(r.getnframes())


def concat_wavs(paths: list[Path]) -> tuple[bytes, wave._wave_params]:
    ref_params = None
    all_frames = []
    for p in paths:
        params, frames = wav_params_and_frames(p)
        if ref_params is None:
            ref_params = params
        else:
            assert params.framerate == ref_params.framerate
            assert params.nchannels == ref_params.nchannels
            assert params.sampwidth == ref_params.sampwidth
        all_frames.append(frames)
    return b"".join(all_frames), ref_params


# Pick a strict_audit participant who actually has q01, q70, q71.
have_all = set.intersection(
    *(set(clips.loc[clips["question_id"] == q, "participant_id"]) for q in (1, 70, 71))
)
demo_pid = sorted(have_all & set(splits.loc[splits["in_strict_audit"], "participant_id"]))[0]
demo_clips = clips[clips["participant_id"] == demo_pid]
demo_paths = [demo_clips.loc[demo_clips["question_id"] == q, "path"].iloc[0] for q in (1, 70, 71)]

input_durations = []
for p in demo_paths:
    params, frames = wav_params_and_frames(p)
    n_frames = len(frames) // (params.sampwidth * params.nchannels)
    input_durations.append(n_frames / params.framerate)
    print(f"  {p.name:30s}  {n_frames:7d} frames  {n_frames/params.framerate:6.3f}s")

concat_frames, params = concat_wavs(demo_paths)
concat_n_frames = len(concat_frames) // (params.sampwidth * params.nchannels)
concat_duration = concat_n_frames / params.framerate

print(f"\nassembled: {concat_n_frames} frames  {concat_duration:.3f}s")
print(f"sum of inputs: {sum(input_durations):.3f}s")
print(f"canonical wav params preserved: {params.framerate} Hz / {params.nchannels} ch / {params.sampwidth*8}-bit")

assert concat_n_frames == sum(
    len(wav_params_and_frames(p)[1]) // (params.sampwidth * params.nchannels) for p in demo_paths
), "Assembled duration differs from sum of inputs — silence or trimming was applied."
print("\n✓ assembled duration equals exact sum of input durations (no silence inserted, no trimming)")

  P0001_q01.wav                    195840 frames  12.240s
  P0001_q70.wav                    113280 frames   7.080s
  P0001_q71.wav                     97920 frames   6.120s

assembled: 407040 frames  25.440s
sum of inputs: 25.440s
canonical wav params preserved: 16000 Hz / 1 ch / 16-bit

✓ assembled duration equals exact sum of input durations (no silence inserted, no trimming)


## 3. Evaluation protocol

Of the 74 prompts in `prompts.csv`, the paper's disparity analysis evaluates only:

- the **55 quantitative scenario prompts** (any scenario row with `is_quantitative=True`), and
- the **demographic-perception probe** (`q72`) and the **dialect/accent-perception probe** (`q73`).

Qualitative scenario prompts (the 14 rows with `prompt_type='scenario'` and `is_quantitative=False`) are excluded from the disparity analysis: they elicit open-ended phrases (a dish, a major, …) for which we do not have a comparable disparity-metric pipeline. The two standalone suffix prompts (`q70`, `q71`) and the microphone check are administrative and likewise not evaluated.

Evaluation is restricted to the **strict_audit** split (500 participants, see notebook 03).

In [5]:
is_quant_scenario = (prompts["prompt_type"] == "scenario") & prompts["is_quantitative"]
is_probe = prompts["question_id"].isin([QID_DEMOGRAPHIC, QID_DIALECT])

evaluable_qids = set(prompts.loc[is_quant_scenario | is_probe, "question_id"].astype(int))
print(f"Evaluable prompt IDs: {len(evaluable_qids)} "
      f"({is_quant_scenario.sum()} quant scenarios + {is_probe.sum()} probes)")

non_evaluable = prompts[~(is_quant_scenario | is_probe)]
print(f"Non-evaluable prompts ({len(non_evaluable)}):")
non_evaluable.groupby("prompt_type").size().to_frame("n_prompts")

Evaluable prompt IDs: 57 (55 quant scenarios + 2 probes)
Non-evaluable prompts (17):


,n_prompts
prompt_type,
mic_check,1
scenario,14
suffix_additional,1
suffix_number,1


Within the strict_audit split, the clips on disk that match evaluable prompts break down as follows. *Before* applying any missing-suffix drops:

In [6]:
strict_pids = set(splits.loc[splits["in_strict_audit"], "participant_id"])
strict_clips = clips[clips["participant_id"].isin(strict_pids)].copy()

# Three evaluable sub-buckets.
both_suffix_qids = set(prompts.loc[
    (prompts["prompt_type"] == "scenario") &
    prompts["suffix_number"] & prompts["suffix_additional"],
    "question_id",
].astype(int))
add_only_quant_qids = set(prompts.loc[
    (prompts["prompt_type"] == "scenario") &
    prompts["is_quantitative"] & (~prompts["suffix_number"]),
    "question_id",
].astype(int))
probe_qids = {QID_DEMOGRAPHIC, QID_DIALECT}

buckets = pd.DataFrame({
    "bucket": [
        "quant scenario, both suffixes needed (assemble q + q70 + q71)",
        "quant scenario, suffix_additional only (assemble q + q71)",
        "demographic probe q72 (standalone)",
        "dialect probe q73 (standalone)",
    ],
    "n_prompts": [
        len(both_suffix_qids),
        len(add_only_quant_qids),
        1,
        1,
    ],
    "n_clips_on_disk_in_strict": [
        len(strict_clips[strict_clips["question_id"].isin(both_suffix_qids)]),
        len(strict_clips[strict_clips["question_id"].isin(add_only_quant_qids)]),
        len(strict_clips[strict_clips["question_id"] == QID_DEMOGRAPHIC]),
        len(strict_clips[strict_clips["question_id"] == QID_DIALECT]),
    ],
})
buckets.loc["total"] = ["TOTAL", buckets["n_prompts"].sum(), buckets["n_clips_on_disk_in_strict"].sum()]
buckets

,bucket,n_prompts,n_clips_on_disk_in_strict
0,"quant scenario, both suffixes needed (assemble...",52,12819
1,"quant scenario, suffix_additional only (assemb...",3,743
2,demographic probe q72 (standalone),1,480
3,dialect probe q73 (standalone),1,489
total,TOTAL,57,14531


## 4. Missing-suffix audit and clip drops

Missing suffix recordings come from intermittent failures of the browser recording widget: participants did reach the suffix prompts, but the captured wav was not recoverable from the upload pipeline. This is therefore not a participant drop-out signal — the participant is otherwise complete — and the affected wavs are not present anywhere in the raw upstream data.

We surface this as two related counts:

1. **Participants in strict_audit who are missing each suffix recording**, and
2. **Clips that cannot be assembled because the participant's suffix is missing.**

In [7]:
have_q70 = set(clips.loc[clips["question_id"] == QID_SUFFIX_NUMBER,     "participant_id"])
have_q71 = set(clips.loc[clips["question_id"] == QID_SUFFIX_ADDITIONAL, "participant_id"])
miss_q70 = strict_pids - have_q70
miss_q71 = strict_pids - have_q71

print(f"strict_audit participants missing q70 (suffix_number):     {len(miss_q70)}")
print(f"strict_audit participants missing q71 (suffix_additional): {len(miss_q71)}")
print(f"strict_audit participants missing BOTH:                    {len(miss_q70 & miss_q71)}")
print(f"strict_audit participants missing AT LEAST ONE:            {len(miss_q70 | miss_q71)}")

strict_audit participants missing q70 (suffix_number):     19
strict_audit participants missing q71 (suffix_additional): 18
strict_audit participants missing BOTH:                    0
strict_audit participants missing AT LEAST ONE:            37


In [8]:
# Per-clip drop logic: keep a clip iff every suffix recording its assembly needs is present.
def assembly_blocked_for(qid: int, pid: str) -> str | None:
    need_q70 = qid in both_suffix_qids
    need_q71 = qid in both_suffix_qids or qid in add_only_quant_qids
    missing = []
    if need_q70 and pid in miss_q70:
        missing.append("q70")
    if need_q71 and pid in miss_q71:
        missing.append("q71")
    return "+".join(missing) if missing else None

quant_clips_strict = strict_clips[
    strict_clips["question_id"].isin(both_suffix_qids | add_only_quant_qids)
].copy()
quant_clips_strict["blocked_by"] = quant_clips_strict.apply(
    lambda r: assembly_blocked_for(int(r["question_id"]), r["participant_id"]), axis=1,
)

drop_summary = (
    quant_clips_strict.groupby(quant_clips_strict["blocked_by"].fillna("— kept —"))
    .size()
    .rename("n_clips")
    .to_frame()
)
drop_summary

,n_clips
blocked_by,
q70,357
q71,362
— kept —,12843


In [9]:
dropped = quant_clips_strict[quant_clips_strict["blocked_by"].notna()]
n_dropped_clips = len(dropped)
n_dropped_pids = dropped["participant_id"].nunique()
print(f"Total quant-scenario clips DROPPED (cannot assemble in participant's own voice): {n_dropped_clips}")
print(f"Unique strict_audit participants whose quant clips are affected:                  {n_dropped_pids}")
assert n_dropped_clips == 719, n_dropped_clips  # the headline number in the paper

Total quant-scenario clips DROPPED (cannot assemble in participant's own voice): 719
Unique strict_audit participants whose quant clips are affected:                  37


**A separate, smaller source of missingness**: the demographic and dialect probes (`q72`, `q73`) are evaluated standalone, so the missing-suffix logic does not apply to them. They are simply *absent* for any participant whose probe recording itself never made it through the upload pipeline. We surface this so it isn't double-counted against the 719:

In [10]:
n_missing_q72 = len(strict_pids - set(clips.loc[clips["question_id"] == QID_DEMOGRAPHIC, "participant_id"]))
n_missing_q73 = len(strict_pids - set(clips.loc[clips["question_id"] == QID_DIALECT,     "participant_id"]))
print(f"strict_audit participants missing demographic probe (q72): {n_missing_q72}")
print(f"strict_audit participants missing dialect probe (q73):     {n_missing_q73}")
print(f"  → standalone probe clips simply not present: {n_missing_q72 + n_missing_q73} "
      f"({n_missing_q72} demographic + {n_missing_q73} dialect)")

strict_audit participants missing demographic probe (q72): 20
strict_audit participants missing dialect probe (q73):     11
  → standalone probe clips simply not present: 31 (20 demographic + 11 dialect)


## 5. The eval set, in one table

Putting it all together: the disparity analysis runs on the clips below. Each row is a kind of evaluable clip; columns split clips into *kept* vs. *not present after assembly*.

In [11]:
kept_quant = quant_clips_strict[quant_clips_strict["blocked_by"].isna()]
kept_both = kept_quant[kept_quant["question_id"].isin(both_suffix_qids)]
kept_add  = kept_quant[kept_quant["question_id"].isin(add_only_quant_qids)]
kept_q72  = strict_clips[strict_clips["question_id"] == QID_DEMOGRAPHIC]
kept_q73  = strict_clips[strict_clips["question_id"] == QID_DIALECT]

dropped_both = dropped[dropped["question_id"].isin(both_suffix_qids)]
dropped_add  = dropped[dropped["question_id"].isin(add_only_quant_qids)]
n_absent_q72 = n_missing_q72
n_absent_q73 = n_missing_q73

eval_table = pd.DataFrame({
    "assembly": [
        "quant scenario   q + q70 + q71",
        "quant scenario   q + q71",
        "demographic probe q72  (standalone)",
        "dialect probe    q73  (standalone)",
    ],
    "n_prompts": [len(both_suffix_qids), len(add_only_quant_qids), 1, 1],
    "clips_kept_in_eval": [len(kept_both), len(kept_add), len(kept_q72), len(kept_q73)],
    "clips_excluded": [len(dropped_both), len(dropped_add), n_absent_q72, n_absent_q73],
    "exclusion_reason": [
        "missing q70 and/or q71",
        "missing q71",
        "probe wav not recoverable from upload pipeline",
        "probe wav not recoverable from upload pipeline",
    ],
})
eval_table.loc["total"] = [
    "TOTAL",
    eval_table["n_prompts"].sum(),
    eval_table["clips_kept_in_eval"].sum(),
    eval_table["clips_excluded"].sum(),
    "",
]
eval_table

,assembly,n_prompts,clips_kept_in_eval,clips_excluded,exclusion_reason
0,quant scenario q + q70 + q71,52,12119,700,missing q70 and/or q71
1,quant scenario q + q71,3,724,19,missing q71
2,demographic probe q72 (standalone),1,480,20,probe wav not recoverable from upload pipeline
3,dialect probe q73 (standalone),1,489,11,probe wav not recoverable from upload pipeline
total,TOTAL,57,13812,750,


**Headline figures (strict_audit split, after assembly):**

- **13,812 clips** evaluated by the model and entering the disparity analysis.
- **719 quantitative-scenario clips excluded** (cannot be assembled in the participant's own voice because at least one of their suffix recordings was lost in the upload pipeline). These remain in the released dataset.
- **31 standalone probe clips never present** (20 demographic, 11 dialect) — those participants never produced a recoverable wav for the probe.
- All released audio is 16 kHz / mono / 16-bit PCM; concatenation is byte-for-byte append with no silence, crossfading, or normalization (sanity-checked in § 2 against a real participant).

Because the standalone scenario, suffix, and probe recordings are released individually, downstream users wishing to apply a different assembly policy (e.g. silence-padded joins, normalized loudness, synthetic suffix substitution) can do so directly from the released files.